In [ ]:
!pip install -q faiss-cpu

In [ ]:
import os
import glob
import torch
import faiss
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor

# Parse data path

In [ ]:
keyframes_dir = "/kaggle/input/keyframes-extracted-data/keyframes_extracted"
all_keyframe_paths = dict()
for part in sorted(os.listdir(keyframes_dir)):
    data_part = part.replace('_extract','') # L21_a, L22_a,..
    all_keyframe_paths[data_part] = dict()

    data_part_path = os.path.join(keyframes_dir, part)
    video_dirs = sorted(os.listdir(data_part_path))
    video_ids = [video_dir.split('_')[-1] for video_dir in video_dirs]
    for video_id, video_dir in zip(video_ids, video_dirs):
        keyframe_paths = sorted(glob.glob(f'{data_part_path}/{video_dir}/*.jpg'))
        all_keyframe_paths[data_part][video_id] = keyframe_paths

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
processor = AutoImageProcessor.from_pretrained("nomic-ai/nomic-embed-vision-v1.5")
vision_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-vision-v1.5", trust_remote_code=True)
vision_model = vision_model.to(device)
vision_model.eval()

In [ ]:
bs = 16
save_dir = './Nomic_features'
if not os.path.exists(save_dir):
  os.mkdir(save_dir)

for key, video_keyframe_paths in tqdm(all_keyframe_paths.items()):
    video_ids = sorted(video_keyframe_paths.keys())
    
    if not os.path.exists(os.path.join(save_dir, key)):
        os.mkdir(os.path.join(save_dir, key))
    
    for video_id in tqdm(video_ids):
        video_feats = []
        video_keyframe_path = video_keyframe_paths[video_id]
        for i in range(0, len(video_keyframe_path), bs):
            # Load images theo batch
            images = []
            image_paths = video_keyframe_path[i:i+bs]
            for image_path in image_paths:
                image = Image.open(image_path).convert("RGB") 
                images.append(image)
            
            # Process images with local model
            inputs = processor(images, return_tensors="pt").to(device)
            
            with torch.no_grad():
                img_emb = vision_model(**inputs).last_hidden_state
                image_feats = F.normalize(img_emb[:, 0], p=2, dim=1)
            
            # Convert to numpy and move to CPU
            image_feats = image_feats.cpu().numpy()
            
            for feat in image_feats:
                video_feats.append(feat.astype(np.float32).flatten()) 
        
        np.save(f'{save_dir}/{key}/{video_id}.npy', video_feats)